In [1]:
import pandas as pd
import numpy as np
import os

np.random.seed(2026)

# 1. Generate Dimension Table: Facilities
facility_data = [
    {"fac_id": "KE-001", "name": "Kenyatta National Hospital", "sub_county": "Dagoretti", "partner": "CDC-Palladium"},
    {"fac_id": "KE-002", "name": "Mbagathi County Hospital", "sub_county": "Kibra", "partner": "CDC-Palladium"},
    {"fac_id": "KE-003", "name": "Mama Lucy Kibaki Hospital", "sub_county": "Embakasi Central", "partner": "USAID-Afya"},
    {"fac_id": "KE-004", "name": "Kisumu County Referral Hosp", "sub_county": "Kisumu Central", "partner": "CHAK"},
    {"fac_id": "KE-005", "name": "JOOTRH Kisumu", "sub_county": "Kisumu Central", "partner": "CDC-Palladium"}
]
df_facilities = pd.DataFrame(facility_data)

# 2. Generate Messy, Wide-Format Fact Table (Longitudinal Track)
cohort_records = []
age_groups = ["15-24", "25-49", "50+"]
genders = ["M", "F"]

for fac in facility_data:
    for age in age_groups:
        for gen in genders:
            # Baseline diagnoses
            diagnosed = np.random.randint(100, 500)
            # Create chaotic, string-formatted text data with extra noise
            initiated = f"{int(diagnosed * np.random.uniform(0.90, 0.98))} cases"
            ret_6mo = int(diagnosed * np.random.uniform(0.75, 0.88))
            # Inject structural logical dependency: 12mo retention cannot exceed 6mo
            ret_12mo = int(ret_6mo * np.random.uniform(0.80, 0.95))
            
            cohort_records.append({
                "MFL_Code": fac["fac_id"] if np.random.rand() > 0.15 else fac["fac_id"].lower().replace("-", ""), # Dirty ID links
                "Demographics": f"{gen} | {age}", # Combined categorical column needing parsing
                "Cohort_Start_Quarter": "2025-Q1",
                "Diagnosed_Count": diagnosed,
                "Initiated_On_ART": initiated,
                "Retained_6_Months": ret_6mo,
                "Retained_12_Months": ret_12mo
            })

df_cohorts_raw = pd.DataFrame(cohort_records)

# Inject dirty numeric text fields with commas and negative flags
df_cohorts_raw.loc[4, "Diagnosed_Count"] = -99  # Out of bounds anomaly
df_cohorts_raw.loc[12, "Retained_12_Months"] = 9999 # Typo outlier

# Export raw, un-cleansed targets
df_facilities.to_csv("dim_facilities_raw.csv", index=False)
df_cohorts_raw.to_excel("fact_patient_cohorts_raw.xlsx", index=False)
print("Complex raw database inputs generated successfully.")

Complex raw database inputs generated successfully.


In [9]:
import pandas as pd
import numpy as np

# Load raw assets
df_fac = pd.read_csv("dim_facilities_raw.csv")
df_coh = pd.read_excel("fact_patient_cohorts_raw.xlsx")

audit_logs = []

# 1. Standardize Primary / Foreign Key links across entities
df_coh['MFL_Code'] = df_coh['MFL_Code'].str.upper().str.strip()
df_coh['MFL_Code'] = df_coh['MFL_Code'].apply(lambda x: f"KE-{x[2:]}" if (len(x)==5 and not "-" in x) else x)

# 2. Extract split variables into structured dimensions
df_coh[['Gender', 'Age_Group']] = df_coh['Demographics'].str.split(' | ', expand=True)[[0, 2]]
df_coh.drop(columns=['Demographics'], inplace=True)

# 3. Structural numerical type conversions and cleansing text markers
df_coh['Initiated_On_ART'] = df_coh['Initiated_On_ART'].str.replace(" cases", "").astype(int)

# 4. Outlier detection, suppression, and logging logic
# Trap Negative Values
neg_mask = df_coh['Diagnosed_Count'] < 0
if neg_mask.any():
    audit_logs.append({"issue": "Negative Diagnosed_Count value detected", "count": neg_mask.sum()})
    df_coh.loc[neg_mask, 'Diagnosed_Count'] = np.nan

# Trap logical impossible bounds (12-Month retention cannot physically exceed 6-Month tracking)
bound_mask = df_coh['Retained_12_Months'] > df_coh['Retained_6_Months']
if bound_mask.any():
    audit_logs.append({"issue": "Logical inversion: 12Mo retention > 6Mo retention", "count": bound_mask.sum()})
    
    # FIX: Explicitly round and cast to integers to prevent the dtype warning
    imputed_values = (df_coh.loc[bound_mask, 'Retained_6_Months'] * 0.85).round().astype('int64')
    df_coh.loc[bound_mask, 'Retained_12_Months'] = imputed_values

# Clean out any structural missing pieces
df_coh['Diagnosed_Count'] = df_coh['Diagnosed_Count'].fillna(df_coh['Diagnosed_Count'].median()).astype(int)
df_coh['Retained_12_Months'] = df_coh['Retained_12_Months'].astype(int)

# Save intermediate production data
df_fac.to_csv("dim_facilities_clean.csv", index=False)
df_coh.to_csv("fact_patient_cohorts_clean.csv", index=False)
pd.DataFrame(audit_logs).to_csv("fact_pipeline_audit.csv", index=False)
print("Data transformation engine run executed cleanly.")

# Save clean intermediate production backups to your local folder
df_fac.to_csv("dim_facilities_clean.csv", index=False)
df_coh.to_csv("fact_patient_cohorts_clean.csv", index=False)

print("Data transformation engine run executed cleanly and local files saved!")

Data transformation engine run executed cleanly.
Data transformation engine run executed cleanly and local files saved!


In [8]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Load the clean dataframes
df_fac = pd.read_csv("dim_facilities_clean.csv")
df_coh = pd.read_csv("fact_patient_cohorts_clean.csv")

# --- THE AUTOMATED FIX: Make ALL column names lowercase to match SQL exactly ---
df_fac.columns = df_fac.columns.str.lower()
df_coh.columns = df_coh.columns.str.lower()
# This effortlessly changes "Cohort_Start_Quarter" to "cohort_start_quarter", etc.
# -----------------------------------------------------------------------------

# 2. Database Connection
DB_USER = "postgres"
DB_PASS = "e^Yd4_g"  # <-- Your password here
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "palladium_kehmis"

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# 3. Safely clear old data
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE fact_patient_cohorts, dim_facilities CASCADE;"))
    conn.commit()
    print("Database tables wiped cleanly. Ready for fresh data insertion.")

# 4. Export data to PostgreSQL
try:
    # Load the Facilities Dimension Table First
    df_fac.to_sql(
        name="dim_facilities", 
        con=engine, 
        if_exists="append",  
        index=False
    )
    print("1. [dim_facilities] populated successfully.")

    # Load the Patient Cohorts Fact Table Second
    df_coh.to_sql(
        name="fact_patient_cohorts", 
        con=engine, 
        if_exists="append", 
        index=False
    )
    print("2. [fact_patient_cohorts] populated successfully.")
    print("\n🎉 Success! Data Warehouse successfully updated with junior relational architecture!")

except Exception as e:
    print(f"\nAn error occurred during loading: {e}")

Database tables wiped cleanly. Ready for fresh data insertion.
1. [dim_facilities] populated successfully.
2. [fact_patient_cohorts] populated successfully.

🎉 Success! Data Warehouse successfully updated with junior relational architecture!
